# 📒 OPERAÇÃO BOLETIM
### Simulado Tabular 1 · Trilha de Dados Tabulares · preparação para a OIAA 2026

---

> **"O boletim do primeiro bimestre chega em abril. A recuperação só começa em novembro.
> No meio do caminho eu perco justamente quem já estava dando sinal."**
>
> — Otávio Ramos, coordenador pedagógico

## 📨 O pedido da coordenação

**De:** Otávio Ramos · Coordenação Pedagógica do Ensino Médio
**Para:** Equipe de Análise de Dados

> Coordeno o ensino médio de uma escola pública há onze anos. Todo fim de ano eu faço a
> mesma conta: quantos estudantes reprovaram que eu poderia ter chamado em maio.
>
> Conseguimos, por um convênio de pesquisa, uma base **real e pública**: mais de mil
> registros de estudantes de duas escolas públicas de Portugal, nas disciplinas de
> Português e Matemática. Cada linha traz o que a escola sabe no começo do ano: a nota do
> **primeiro bimestre**, as faltas, as reprovações anteriores, a rotina, a família. E traz
> também como o ano terminou.
>
> Transformamos a nota final (de 0 a 20; aprova quem tira 10 ou mais) em três níveis de
> **risco de reprovação**:
>
> - **Baixo** — terminou com 12 ou mais.
> - **Médio** — passou raspando, com 10 ou 11.
> - **Alto** — reprovou, com menos de 10.
>
> Três avisos antes de vocês começarem. **A base é pequena**: com poucas centenas de
> estudantes na validação, o número muda bastante de uma divisão para outra, e eu não
> quero decidir nada em cima de sorte. **O mesmo estudante pode aparecer duas vezes**, uma
> por disciplina. E **a nota do bimestre já diz muito**: se o modelo de vocês só repetir o
> boletim, eu não preciso dele.
>
> — Otávio Ramos

---

*O cenário é simulado; os dados são reais (Cortez & Silva, 2008, UCI Machine Learning
Repository). O conjunto de teste foi separado por estudante: quem está no teste não
aparece no conjunto público.*

## 🎯 A sua missão

Construir um modelo que, olhando o que a escola sabe no começo do ano, aponte o **risco
de reprovação** de cada estudante: **baixo**, **médio** ou **alto**.

O problema é de **classificação supervisionada**: há casos com resposta conhecida, e a
tarefa é induzir uma regra que **generalize** para casos novos. Com uma base pequena,
generalizar é o desafio inteiro: é fácil achar uma configuração que brilha numa divisão
e afunda na seguinte.

### 🗺️ O mapa da operação

| Bloco | O que vocês fazem | Como |
|:---|:---|:---|
| **1 · EXPLORAR** | Conhecer os estudantes e procurar sinais | Só clicando |
| **2 · CONFIGURAR** | Decidir como o modelo vai aprender | Só arrastando |
| **3 · MODELAR** | Construir, treinar e prever | ✏️ Completando **8 lacunas** |
| **4 · AVALIAR** | Medir, testar a estabilidade e comparar com a regra do boletim | Só clicando |

> ⚠️ **Antes de começar:** vá em `Ambiente de execução` → `Alterar o tipo de ambiente de
> execução` e confirme que está em **CPU**. Tudo aqui roda em segundos, sem placa de vídeo.
>
> Use `Arquivo` → `Salvar uma cópia no Drive` antes de começar, para não perder o trabalho.
>
> 📎 **Simulado de preparação.** Este caderno imita o formato dos desafios oficiais da Fase 1,
> com dados públicos diferentes. O gabarito do teste fica com o corretor
> (`simulados/corrigir.py`), nunca dentro do caderno.

In [ ]:
#@title 🛠️ Preparação: abrindo os registros da escola { display-mode: "form" }
# Os arquivos treino.csv e teste.csv ficam na mesma pasta deste caderno.
# No Colab, a célula pede o envio dos dois arquivos.
URL_TREINO = ""
URL_TESTE = ""

import pathlib, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, recall_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)

NOMES = {0: "Baixo", 1: "Médio", 2: "Alto"}
CORES = ["#3DCB55", "#FFC107", "#E6177A"]
ALVO = "risco_reprovacao"

def _carregar(url, nome_local):
    """Usa a cópia ao lado do caderno; senão tenta o endereço; senão pede o envio."""
    if pathlib.Path(nome_local).exists():
        return pd.read_csv(nome_local)
    if url:
        print(f"Baixando {nome_local}...")
        try:
            return pd.read_csv(url)
        except Exception as erro:
            print(f"  não deu certo ({erro.__class__.__name__}).")
    from google.colab import files
    print(f"  envie o arquivo {nome_local}:")
    files.upload()
    return pd.read_csv(nome_local)

# Conjunto PÚBLICO: vocês têm as respostas. Serve para treinar e validar.
dados = _carregar(URL_TREINO, "treino.csv")

# Conjunto de TESTE: tamanho fixo, respostas ocultas (ficam com o corretor).
_teste = _carregar(URL_TESTE, "teste.csv")
IDS_TESTE = _teste["id_estudante"].to_numpy()
dados_teste = _teste.drop(columns=["id_estudante"])
N_TESTE = len(dados_teste)

print("Ambiente carregado.\n")
print(f"  Conjunto público (com respostas) .... {len(dados):>5} registros")
print(f"  Conjunto de teste (sem respostas) ... {N_TESTE:>5} registros  [tamanho fixo]")
print(f"  Variáveis disponíveis ............... {dados.shape[1]-1}\n")
for k, n in NOMES.items():
    q = (dados[ALVO] == k).sum()
    print(f"     risco {n:<6} ({k}) .................. {q:>5}   {q/len(dados)*100:4.1f}%")
print(f"\n  A submissão precisará ter exatamente {N_TESTE} linhas.")
display(dados.head())

---
# 🔍 BLOCO 1 · EXPLORAR
## *O que o começo do ano já conta*

> **Otávio:** *"Olhem os registros antes de treinar qualquer coisa. Eu quero saber o que
> vocês enxergam que eu não enxergo."*

Algumas variáveis do conjunto:

| Variável | O que é |
|:---|:---|
| `nota_1bim` | Nota do primeiro bimestre (0 a 20) |
| `faltas` | Número de faltas no ano |
| `reprovacoes_anteriores` | Quantas vezes já reprovou (0 a 3; 3 significa "3 ou mais") |
| `horas_estudo` | Estudo semanal (1: <2h · 2: 2–5h · 3: 5–10h · 4: >10h) |
| `escolaridade_mae`, `escolaridade_pai` | 0: nenhuma · 1: fundamental I · 2: fundamental II · 3: médio · 4: superior |
| `ocupacao_mae`, `ocupacao_pai` | 0: em casa · 1: serviços · 2: saúde · 3: professor(a) · 4: outra |
| `alcool_dias_semana`, `alcool_fim_de_semana` | Consumo declarado de álcool (1: muito baixo a 5: muito alto) |
| `quer_ensino_superior`, `internet_em_casa`, `reforco_escolar`… | 1 = sim, 0 = não |
| `disciplina_matematica` | 1 = registro de Matemática, 0 = de Português |

Três perguntas para guiar a exploração:

- Qual variável separa melhor os três níveis? É só a nota, ou há sinais que a nota não mostra?
- O nível **médio** fica no meio de tudo, ou tem características próprias?
- Que combinações só indicam risco alto quando aparecem **juntas**?

In [ ]:
#@title 📊 Investigue os dados: escolha a variável E o tipo de gráfico { display-mode: "form" }
#@markdown Escolha a variável e o gráfico, depois rode a célula (▶).
variavel = "nota_1bim"  #@param ["nota_1bim", "faltas", "reprovacoes_anteriores", "idade", "horas_estudo", "escolaridade_mae", "escolaridade_pai", "tempo_deslocamento", "sai_com_amigos", "alcool_fim_de_semana", "quer_ensino_superior", "internet_em_casa", "reforco_escolar", "disciplina_matematica"]
tipo_de_grafico = "Boxplot"  #@param ["Boxplot", "Histograma sobreposto", "Violino", "Barras de proporcao"]

serie = dados[variavel]
poucos_valores = serie.nunique() <= 5
grupos = [serie[dados[ALVO] == k] for k in NOMES]
rot = [NOMES[k] for k in NOMES]

INCOMPAT = {
    ("Histograma sobreposto", True): "a variável tem poucos valores; o histograma vira barras soltas",
    ("Violino", True): "o violino estima uma densidade contínua, que não existe aqui",
    ("Barras de proporcao", False): "proporção por valor exige poucos valores distintos, e aqui há muitos",
}
aviso = INCOMPAT.get((tipo_de_grafico, poucos_valores))
print(f"Variável: {variavel}  |  valores distintos: {serie.nunique()}")
if aviso:
    print(f"\n[!] {tipo_de_grafico} não é a melhor escolha aqui, porque {aviso}.")
    print("    O gráfico será exibido assim mesmo, para vocês verem o problema.\n")

fig, ax = plt.subplots(figsize=(8.5, 4.4))
if tipo_de_grafico == "Boxplot":
    bp = ax.boxplot(grupos, patch_artist=True, tick_labels=rot,
                    medianprops=dict(color="black", linewidth=2))
    for p, cor in zip(bp["boxes"], CORES): p.set_facecolor(cor); p.set_alpha(.8)
    ax.set_ylabel(variavel)
elif tipo_de_grafico == "Histograma sobreposto":
    bins = np.histogram_bin_edges(serie, bins=20)
    for g, cor, r in zip(grupos, CORES, rot):
        ax.hist(g, bins=bins, alpha=.55, color=cor, label=r, edgecolor="white", density=True)
    ax.legend(); ax.set_xlabel(variavel); ax.set_ylabel("densidade")
elif tipo_de_grafico == "Violino":
    vp = ax.violinplot(grupos, showmedians=True)
    for corpo, cor in zip(vp["bodies"], CORES): corpo.set_facecolor(cor); corpo.set_alpha(.75)
    ax.set_xticks([1, 2, 3]); ax.set_xticklabels(rot); ax.set_ylabel(variavel)
else:
    tab = pd.crosstab(serie, dados[ALVO], normalize="index")[list(NOMES)]
    tab.columns = rot
    tab.plot(kind="bar", stacked=True, color=CORES, ax=ax, edgecolor="white")
    ax.set_ylabel("proporção dentro de cada valor"); ax.legend(title="risco", bbox_to_anchor=(1, 1))
ax.set_title(f"'{variavel}' por nível de risco", fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

print("PERGUNTAS:")
print("   Os três níveis se separam, ou as caixas se sobrepõem quase inteiras?")
print("   O nível MÉDIO se distingue do BAIXO, ou só do ALTO?")
print("   Barras de proporção: em que valor a fatia de risco ALTO dispara?")

In [ ]:
#@title 🧩 Combinações: dois sinais ao mesmo tempo { display-mode: "form" }
#@markdown Escolha dois sinais. A tabela mostra o % de risco ALTO em cada combinação.
sinal_1 = "reprovacoes_anteriores"  #@param ["reprovacoes_anteriores", "horas_estudo", "quer_ensino_superior", "internet_em_casa", "reforco_escolar", "disciplina_matematica", "escola_b", "genero"]
sinal_2 = "disciplina_matematica"  #@param ["disciplina_matematica", "reprovacoes_anteriores", "horas_estudo", "quer_ensino_superior", "internet_em_casa", "reforco_escolar", "escola_b", "genero"]

if sinal_1 == sinal_2:
    print("Escolham dois sinais diferentes.")
else:
    pct_alto = pd.crosstab(dados[sinal_1], dados[sinal_2], values=(dados[ALVO] == 2),
                           aggfunc="mean") * 100
    contagem = pd.crosstab(dados[sinal_1], dados[sinal_2])
    print(f"% de risco ALTO  (linhas = {sinal_1}, colunas = {sinal_2})\n")
    display(pct_alto.round(1))
    print("\nQuantos estudantes há em cada combinação:\n")
    display(contagem)
    print("Cuidado: uma porcentagem alta sobre 4 estudantes não é evidência de nada.")
    print("Leiam sempre as duas tabelas juntas.")

---
# ⚙️ BLOCO 2 · CONFIGURAR
## *As decisões que definem o desempenho*

### Como os dados estão organizados

| Conjunto | Tamanho | Vocês têm as respostas? | Para que serve |
|:---|:---|:---|:---|
| **Público** | 783 registros | Sim | Treinar o modelo e validar as escolhas |
| **Teste** | **261 (fixo)** | Não | Comparação final (corrigida por `simulados/corrigir.py`) |

O controle `tamanho_da_validacao` define a fatia do público reservada para validar. Com
uma base deste tamanho, **a própria divisão muda o resultado**: por isso existe também
o controle `semente_da_divisao`. Troquem a semente mantendo todo o resto igual e
observem o quanto o F1-macro oscila. Uma melhora menor que essa oscilação não é
melhora, é ruído.

In [ ]:
#@title 🎛️ Parâmetros do experimento { display-mode: "form" }
#@markdown Ajuste os controles e rode a célula (▶) para registrar a configuração.
tamanho_da_validacao = 25  #@param {type:"slider", min:10, max:40, step:5}
semente_da_divisao = 42  #@param {type:"integer"}
modelo_escolhido = "Arvore de Decisao"  #@param ["Arvore de Decisao", "Floresta Aleatoria", "Regressao Logistica"]
profundidade = 10  #@param {type:"slider", min:2, max:20, step:1}
n_arvores = 100  #@param {type:"slider", min:50, max:600, step:50}

n_valida = int(round(len(dados) * tamanho_da_validacao / 100))
print("PARÂMETROS DEFINIDOS")
print(f"   Validação .......... {tamanho_da_validacao}%  ({n_valida} registros)")
print(f"   Treino ............. {100-tamanho_da_validacao}%  ({len(dados)-n_valida} registros)")
print(f"   Semente da divisão . {semente_da_divisao}")
print(f"   Modelo ............. {modelo_escolhido}")
if modelo_escolhido != "Regressao Logistica":
    print(f"   Profundidade ....... {profundidade}")
if modelo_escolhido == "Floresta Aleatoria":
    print(f"   Árvores ............ {n_arvores}")
print(f"   Teste .............. {N_TESTE} registros  [FIXO]")
if modelo_escolhido != "Regressao Logistica" and profundidade >= 10:
    print("\n[!] Profundidade alta numa base de 600 registros de treino. Comparem o")
    print("    F1 de TREINO com o de VALIDAÇÃO no Bloco 4 antes de comemorar.")

### As decisões e seus efeitos

**Profundidade.** Uma árvore profunda cria regras cada vez mais específicas até isolar
estudantes individuais. Com 600 exemplos de treino, isso acontece rápido: o modelo decora
o treino e erra na validação (**sobreajuste**). Uma árvore rasa demais não captura nem o
óbvio (**subajuste**).

**Desequilíbrio entre as classes.** Metade dos registros é de risco baixo. O nível médio
é o mais difícil: fica espremido entre os outros dois. Existe um parâmetro nas tabelas
abaixo que muda o peso de cada classe no treino. Procurem por ele.

**Escala.** A Regressão Logística soma variáveis multiplicadas por pesos. `faltas` vai de
0 a 75; `internet_em_casa`, de 0 a 1. Isso afeta a otimização e é um dos motivos do
aviso de não convergência. Árvores não se importam com escala.

### 📗 Referência técnica: nome exato e parâmetros de cada classe

**Vocês vão precisar desta tabela no Bloco 3.**

> **Por que isso importa.** No Bloco 3, apenas alguns parâmetros aparecem preenchidos.
> Todos os demais assumem silenciosamente o valor padrão. Completar só o nome da classe
> produz um resultado correto e mediano. Estudar esta tabela e **acrescentar parâmetros
> à chamada** produz resultados melhores. A diferença é intencional.

#### 🌳 Árvore de Decisão

```python
from sklearn.tree import DecisionTreeClassifier
```

| Parâmetro | Padrão | O que faz |
|:---|:---|:---|
| `max_depth` | `None` | Profundidade máxima. `None` cresce até separar tudo — e decora o treino |
| `min_samples_leaf` | `1` | Mínimo de exemplos por folha. Aumentar suaviza as regras |
| `min_samples_split` | `2` | Mínimo de exemplos para tentar um novo corte |
| `criterion` | `'gini'` | Medida de impureza. Alternativa: `'entropy'` |
| `class_weight` | `None` | `'balanced'` dá mais peso às classes raras |
| `random_state` | `None` | Semente, para o resultado ser reproduzível |

#### 🌲 Floresta Aleatória

```python
from sklearn.ensemble import RandomForestClassifier
```

| Parâmetro | Padrão | O que faz |
|:---|:---|:---|
| `n_estimators` | `100` | Número de árvores |
| `max_depth` | `None` | Profundidade máxima de cada árvore |
| `min_samples_leaf` | `1` | Mínimo de exemplos por folha |
| `max_features` | `'sqrt'` | Variáveis sorteadas em cada corte. Aceita fração |
| `class_weight` | `None` | `'balanced'` ou `'balanced_subsample'` |
| `n_jobs` | `None` | `-1` usa todos os núcleos |
| `random_state` | `None` | Semente |

#### 📈 Regressão Logística

```python
from sklearn.linear_model import LogisticRegression
```

| Parâmetro | Padrão | O que faz |
|:---|:---|:---|
| `C` | `1.0` | Inverso da força da regularização. **Menor regulariza mais** |
| `penalty` | `'l2'` | Tipo de regularização |
| `max_iter` | `100` | Iterações máximas. **Frequentemente insuficiente**: o aviso de não convergência passa despercebido |
| `class_weight` | `None` | `'balanced'` compensa o desequilíbrio entre classes |
| `random_state` | `None` | Semente |

---
# ✏️ BLOCO 3 · MODELAR
## *Aqui vocês escrevem código*

### A interface comum dos estimadores

| Método | Operação | Onde se aplica |
|:---|:---|:---|
| `.fit(X, y)` | **Ajusta** os parâmetros internos a partir dos exemplos e das respostas | Só no treino |
| `.predict(X)` | **Infere** a classe de casos novos, com os parâmetros já ajustados | Validação e teste |

**`X`** é a tabela de variáveis; **`y`**, o vetor de respostas. Sem `fit`, não há o que
`predict` usar.

### Sobre a estrutura condicional

O trecho abaixo escolhe um entre três modelos conforme o Bloco 2, com uma cadeia
`if` / `elif` / `else`. O `else` não recebe condição, apenas os dois-pontos.

> ⚠️ **A célula não executa enquanto houver lacunas.** Preencham primeiro, executem depois.

In [ ]:
# ✏️ SUA VEZ · Parte 1: separar os dados e construir o modelo
# Substituam cada ____ pelo que falta. A célula só executa quando não sobrar nenhum.
X = dados.drop(columns=[ALVO])
y = dados[ALVO]

# LACUNA 1 ▸ 'stratify' precisa receber o vetor cuja proporção deve ser preservada
#            nas duas partes. É a tabela de variáveis ou o vetor de respostas?
X_treino, X_valida, y_treino, y_valida = train_test_split(
    X, y, test_size=tamanho_da_validacao/100, random_state=semente_da_divisao,
    stratify=____)

# LACUNAS 2 a 6 ▸ os nomes exatos das classes estão na referência do Bloco 2.
#                 Faltam também as duas palavras que continuam a cadeia condicional.
if modelo_escolhido == "Arvore de Decisao":
    modelo = ____(max_depth=profundidade, random_state=42)

____ modelo_escolhido == "Floresta Aleatoria":
    modelo = ____(n_estimators=n_arvores, max_depth=profundidade,
                  random_state=42, n_jobs=-1)

____:
    modelo = ____(random_state=42)

# ZONA LIVRE ▸ vocês podem acrescentar parâmetros às chamadas acima.
#   As tabelas do Bloco 2 listam os disponíveis e o valor padrão de cada um. Dois
#   merecem atenção: um compensa o desequilíbrio entre os níveis de risco, e outro
#   (só na Regressão Logística) evita que o treino pare antes de convergir.
#   Mudem UM de cada vez e registrem o efeito sobre o F1-macro no Bloco 4.

print(f"Treino:    {len(X_treino)} registros")
print(f"Validação: {len(X_valida)} registros")
print(f"Modelo construído: {type(modelo).__name__}")

In [ ]:
# ✏️ SUA VEZ · Parte 2: treinar e prever
# LACUNA 7 ▸ qual método ajusta os parâmetros do modelo ao conjunto de treino?
modelo.____(X_treino, y_treino)

# LACUNA 8 ▸ qual método infere as classes de casos que o modelo nunca viu?
previsoes = modelo.____(X_valida)

print(f"Modelo ajustado. {len(previsoes)} previsões geradas para a validação.")

In [ ]:
#@title 🔎 Verificação das lacunas { display-mode: "form" }
erros = []
try:
    # Um split estratificado reproduz a composição de cada classe com erro de no
    # máximo um exemplo. Sem stratify, o desvio costuma ser bem maior.
    esperado = np.bincount(y, minlength=3) * len(y_treino) / len(y)
    if np.abs(np.bincount(y_treino, minlength=3) - esperado).max() > 1.5:
        erros.append("LACUNA 1: as proporções dos níveis no treino diferem do conjunto "
                     "completo. O 'stratify' não recebeu o vetor correto.")
except NameError:
    erros.append("LACUNA 1: X_treino/y_treino não existem. Rode a Parte 1.")
try:
    esperado = {"Arvore de Decisao": "DecisionTreeClassifier",
                "Floresta Aleatoria": "RandomForestClassifier",
                "Regressao Logistica": "LogisticRegression"}[modelo_escolhido]
    if type(modelo).__name__ != esperado:
        erros.append(f"LACUNAS 2 a 6: para '{modelo_escolhido}' era esperada a classe "
                     f"{esperado}, mas foi construída {type(modelo).__name__}. Revisem os "
                     "nomes e a cadeia if / elif / else.")
except NameError:
    erros.append("LACUNAS 2 a 6: a variável 'modelo' não existe. Rode a Parte 1.")
try:
    if not hasattr(modelo, "classes_"):
        erros.append("LACUNA 7: o modelo não possui parâmetros ajustados. Qual método "
                     "faz o ajuste aos dados de treino?")
except NameError: pass
try:
    if len(previsoes) != len(X_valida):
        erros.append(f"LACUNA 8: {len(previsoes)} previsões para {len(X_valida)} casos.")
except NameError:
    erros.append("LACUNA 8: a variável 'previsoes' não existe.")

if erros:
    print("VERIFICAÇÃO REPROVADA. Pontos a revisar:\n")
    for e in erros: print("   [!] " + e + "\n")
else:
    print("VERIFICAÇÃO APROVADA\n")
    print("   Dados estratificados, modelo construído e ajustado, previsões geradas.")
    extras = {k: v for k, v in modelo.get_params().items()
              if k in ("class_weight", "min_samples_leaf", "max_features", "C",
                       "max_iter", "criterion", "min_samples_split")
              and v not in (None, 1, 2, "gini", "sqrt", 1.0, 100)}
    if extras:
        print(f"\n   Parâmetros adicionais detectados na ZONA LIVRE: {extras}")
    else:
        print("\n   Nenhum parâmetro adicional declarado: o modelo usa os valores padrão.")
        print("   É válido, mas as tabelas do Bloco 2 oferecem margem real de melhoria.")

---
# 📈 BLOCO 4 · AVALIAR
## *O modelo enxerga o que o boletim não mostra?*

> **Otávio:** *"Três coisas: quanto acerta, como erra, e se isso se sustenta quando eu
> troco a amostra."*

A métrica oficial é o **F1-macro**: o F1 de cada nível, com média simples entre os
três. Um modelo que chamasse todo mundo de "baixo" acertaria metade dos casos, com F1-macro
perto de 0,22. A métrica de apoio aqui é a **revocação do risco alto**: dos estudantes que
de fato reprovaram, quantos o modelo apontou a tempo.

In [ ]:
#@title 🏆 Desempenho na validação { display-mode: "form" }
acuracia = accuracy_score(y_valida, previsoes)
f1 = f1_score(y_valida, previsoes, average="macro")
recall_alto = recall_score(y_valida, previsoes, labels=[2], average="macro")
f1_treino = f1_score(y_treino, modelo.predict(X_treino), average="macro")

print("="*60)
print("            DESEMPENHO NO CONJUNTO DE VALIDAÇÃO")
print("="*60)
print(f"  Acurácia ............................. {acuracia*100:5.1f}%")
print(f"  ⭐ F1-MACRO (métrica oficial) ......... {f1*100:5.1f}%")
print(f"  🔴 Revocação do risco ALTO ............ {recall_alto*100:5.1f}%")
print(f"     F1-macro no próprio TREINO ......... {f1_treino*100:5.1f}%")
print("="*60)
# Faixas calibradas neste conjunto (validação de 25%, semente 42).
if   f1 >= .72: print("  Desempenho alto. Confirmem na célula de estabilidade antes de comemorar.")
elif f1 >= .66: print("  Desempenho sólido, com margem no Bloco 2 e na ZONA LIVRE.")
elif f1 >= .60: print("  Desempenho intermediário. Revisem profundidade e peso das classes.")
else:           print("  Desempenho baixo. A configuração atual não captura os padrões.")
if f1_treino - f1 > .15:
    print(f"\n[!] O treino está {100*(f1_treino-f1):.0f} pontos acima da validação: sinal clássico")
    print("    de sobreajuste. O modelo decorou estudantes em vez de aprender padrões.")

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix(y_valida, previsoes, labels=[0, 1, 2]),
    display_labels=list(NOMES.values())).plot(ax=ax, cmap="viridis", colorbar=False)
ax.set_title("Matriz de confusão (validação)", fontweight="bold")
ax.set_xlabel("O que o modelo DISSE"); ax.set_ylabel("O que REALMENTE aconteceu")
plt.tight_layout(); plt.show()

cm = confusion_matrix(y_valida, previsoes, labels=[0, 1, 2])
print(f"\n  Reprovados que o modelo chamou de risco BAIXO ... {cm[2,0]}  (o erro mais caro)")
print(f"  Reprovados chamados de risco MÉDIO .............. {cm[2,1]}")
print(f"  Risco baixo chamados de ALTO .................... {cm[0,2]}\n")
print(classification_report(y_valida, previsoes, labels=[0, 1, 2],
      target_names=list(NOMES.values()), digits=3))

In [ ]:
#@title 🎲 O resultado se sustenta? Estabilidade em cinco divisões { display-mode: "form" }
# Repete EXATAMENTE a configuração atual (mesmo modelo, mesmos parâmetros) em cinco
# divisões diferentes do conjunto público. Só a semente da divisão muda.
from sklearn.base import clone
resultados = []
for s in [1, 7, 42, 123, 2026]:
    Xt, Xv, yt, yv = train_test_split(X, y, test_size=tamanho_da_validacao/100,
                                      random_state=s, stratify=y)
    m = clone(modelo).fit(Xt, yt)
    resultados.append(f1_score(yv, m.predict(Xv), average="macro"))
resultados = np.array(resultados)
print("F1-macro em cinco divisões:", "  ".join(f"{r*100:.1f}" for r in resultados))
print(f"\n   Média ........ {resultados.mean()*100:5.1f}%")
print(f"   Oscilação .... ±{resultados.std()*100:4.1f} pontos (desvio-padrão)")
print(f"   Pior / melhor  {resultados.min()*100:.1f}% / {resultados.max()*100:.1f}%")
print("\nAo comparar duas configurações, comparem as MÉDIAS. Uma diferença menor que a")
print("oscilação acima pode ter vindo só da sorte na divisão.")

In [ ]:
#@title 🧮 O modelo bate a regra de bolso do boletim? { display-mode: "form" }
# A coordenação já tem uma regra: nota do 1º bimestre abaixo de 10 → risco alto;
# 10 ou 11 → médio; 12 ou mais → baixo. Nenhum aprendizado envolvido.
regra = np.select([X_valida["nota_1bim"] < 10, X_valida["nota_1bim"] < 12], [2, 1], 0)
f1_regra = f1_score(y_valida, regra, average="macro")
rec_regra = recall_score(y_valida, regra, labels=[2], average="macro")
print(f"                       F1-macro   Revocação do ALTO")
print(f"   Regra do boletim ... {f1_regra*100:6.1f}%      {rec_regra*100:6.1f}%")
print(f"   Modelo de vocês .... {f1*100:6.1f}%      {recall_alto*100:6.1f}%")
if f1 <= f1_regra:
    print("\n[!] O modelo não supera a regra de uma linha. O Otávio foi claro: se for só")
    print("    para repetir o boletim, ele não precisa de modelo. Voltem ao Bloco 2.")
else:
    print(f"\n   O modelo ganha {100*(f1-f1_regra):.1f} pontos da regra. Esse ganho é maior")
    print("   que a oscilação medida na célula anterior? Se não for, não dá para afirmar.")

In [ ]:
#@title 🔦 O que o modelo está enxergando? { display-mode: "form" }
if hasattr(modelo, "feature_importances_"):
    imp = pd.Series(modelo.feature_importances_, index=X.columns).sort_values()
    titulo = "Importância de cada variável na decisão"
else:
    imp = pd.Series(np.abs(modelo.coef_).mean(axis=0), index=X.columns).sort_values()
    titulo = "Peso de cada variável (média dos coeficientes, sem padronização)"
imp = imp.tail(15)
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.barh(imp.index, imp.values, color="#00B8D4", edgecolor="#0A0F2C")
ax.set_title(titulo, fontsize=13, fontweight="bold")
ax.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.show()

print("AS CINCO VARIÁVEIS DE MAIOR PESO:\n")
for i, (nome, val) in enumerate(imp.sort_values(ascending=False).head(5).items(), 1):
    print(f"   {i}. {nome}  ({val:.3f})")
print("\nImportância não é causa. 'faltas' pesar muito não prova que faltar reprova:")
print("pode ser que as duas coisas venham de um terceiro fator (trabalho, doença, família).")
if not hasattr(modelo, "feature_importances_"):
    print("\nNa Regressão Logística sem padronização, o tamanho do coeficiente também")
    print("depende da escala da variável. Comparem com cautela.")

---
# 🤔 PARADA OBRIGATÓRIA: o que a escola faz com a lista

> **Otávio:** *"Uma lista de 'risco alto' pode virar convite para o reforço ou pode
> virar rótulo na sala dos professores. O arquivo é o mesmo. O que muda é o que a gente
> faz com ele."*

### Questões para a equipe deliberar

1. **Variáveis sensíveis.** A base traz `alcool_fim_de_semana`, `namorando`, `genero`,
   `escolaridade_mae`, `ocupacao_pai`. Algumas foram *declaradas pelo próprio estudante*
   num questionário. **Testem:** retirem essas colunas de `X` no Bloco 3
   (`dados.drop(columns=[ALVO, ...])`), reexecutem e comparem o F1-macro — lembrando da
   oscilação que vocês mediram. Se a perda for menor que a oscilação, vale a pena manter
   essas variáveis?

2. **Dados de outro país.** Os registros vêm de Portugal, de 2005–2006. Que diferenças de
   sistema escolar, de escala de notas e de contexto social tornariam arriscado usar este
   modelo numa escola brasileira sem retreinar com dados locais?

3. **O mesmo estudante, duas linhas.** No conjunto público, um estudante pode ter um
   registro de Português e outro de Matemática. Se um cair no treino e o outro na
   validação, o que acontece com a validação? (Dica: o teste foi separado por estudante
   justamente por isso.)

4. **Governança.** Quem pode ver a lista? Por quanto tempo ela vale? O que é proibido
   fazer com ela? Quem responde por uma decisão tomada com base no modelo?

In [ ]:
#@title 📝 O relatório da sua equipe para a coordenação { display-mode: "form" }
#@markdown Este texto conta como critério de desempate nos desafios oficiais.
nome_da_equipe = ""  #@param {type:"string"}
#@markdown **1. Qual configuração vocês escolheram, e ela se sustentou na célula de estabilidade?**
resposta_1 = ""  #@param {type:"string"}
#@markdown **2. Quais sinais mais pesam, além da nota do bimestre? O modelo bateu a regra do boletim?**
resposta_2 = ""  #@param {type:"string"}
#@markdown **3. O que aconteceu ao retirar as variáveis sensíveis? Vocês as manteriam?**
resposta_3 = ""  #@param {type:"string"}
#@markdown **4. Que restrições de uso vocês anexariam à entrega?**
resposta_4 = ""  #@param {type:"string"}

relatorio = {"equipe": nome_da_equipe, "modelo": modelo_escolhido,
             "validacao_pct": tamanho_da_validacao, "semente": semente_da_divisao,
             "profundidade": profundidade, "n_arvores": n_arvores,
             "f1_macro": round(f1, 4), "acuracia": round(acuracia, 4),
             "recall_alto": round(recall_alto, 4),
             "just_modelo": resposta_1, "just_sinais": resposta_2,
             "just_sensiveis": resposta_3, "just_restricoes": resposta_4}
faltando = [k for k in ["equipe", "just_modelo", "just_sinais", "just_sensiveis",
                        "just_restricoes"] if not str(relatorio[k]).strip()]
if faltando:
    print("⚠️ Ainda faltam campos:", ", ".join(faltando))
else:
    print("Relatório completo. Prossigam para a submissão.")

In [ ]:
#@title 🚀 Gerar o arquivo de submissão { display-mode: "form" }
#@markdown Rode a célula (▶). Os dois arquivos ficam em submissoes/.
import json, pathlib

PASTA_SUBMISSAO = pathlib.Path("submissoes")
PASTA_SUBMISSAO.mkdir(parents=True, exist_ok=True)
if "relatorio" not in dir() or not isinstance(relatorio, dict):
    relatorio = {"equipe": "", "modelo": modelo_escolhido, "f1_macro": round(f1, 4)}

# O modelo é aplicado ao teste, cujas respostas ficam com o corretor.
palpites = modelo.predict(dados_teste[X.columns])
submissao = pd.DataFrame({"id_estudante": IDS_TESTE, "risco_previsto": palpites})

assert len(submissao) == N_TESTE, f"A submissão precisa ter {N_TESTE} linhas."
csv_saida = PASTA_SUBMISSAO / "submissao_boletim.csv"
json_saida = PASTA_SUBMISSAO / "relatorio_boletim.json"
submissao.to_csv(csv_saida, index=False)
with open(json_saida, "w", encoding="utf-8") as f:
    json.dump(relatorio, f, ensure_ascii=False, indent=2)

print("="*60)
print("             ARQUIVOS PRONTOS PARA ENVIO")
print("="*60)
print(f"  submissao_boletim.csv   ({len(submissao)} linhas)")
print("  relatorio_boletim.json")
print("="*60)
print("\n  Distribuição prevista no teste:")
for k, v in pd.Series(palpites).value_counts().sort_index().items():
    print(f"     {NOMES[k]:<6} {v:>4}  ({v/len(palpites)*100:.1f}%)")
display(submissao.head())
print("\n  Para corrigir (da raiz do repositório):")
print(f"     python simulados/corrigir.py boletim {csv_saida}")

try:
    from google.colab import files
    files.download(str(csv_saida))
    files.download(str(json_saida))
    print("\n  Download iniciado.")
except ImportError:
    print(f"\n  Fora do Colab: os arquivos estão em {PASTA_SUBMISSAO.resolve()}")

---
# 🌟 FIM DA OPERAÇÃO BOLETIM

> **Otávio:** *"Levei a lista para o conselho de classe de maio, com a regra que vocês
> escreveram: quem aparece nela é convidado para o reforço, e ninguém fora da
> coordenação vê o nível de risco de ninguém. Foi a primeira vez que a gente começou a
> recuperação antes de novembro."*

---

### Competências praticadas neste simulado

- Divisão estratificada e o papel da semente na variabilidade da validação
- Sobreajuste em bases pequenas: comparar F1 de treino e de validação
- Comparar com uma **regra simples** antes de defender um modelo
- Peso das classes e o nível intermediário espremido entre os extremos
- Vazamento por registros repetidos da mesma pessoa
- Variáveis sensíveis: testar a remoção em vez de supor o efeito
- Governança de listas de risco aplicadas a pessoas

**Otimização.** Um fator por vez, sempre olhando a **média** da célula de estabilidade.